# CARVE on CARE-PD — 3D pose, UPDRS-gait severity

**Before running past Section 1**: CARE-PD's exact post-`preprocess_smpl2h36m.sh` file structure (dict keys, array shapes, how UPDRS scores are attached) isn't something I can verify without having actually run their pipeline — this notebook is built from their documented repo structure and the standard H36M 17-joint convention, but Section 1 is an **inspection step**, not an assumption. Run it, look at the printed output, and adjust Section 2's loader to match what you actually see before trusting anything past that point — the same approach used earlier in this project for the LA and rat-movement JSON files, whose real structure also had to be discovered rather than guessed.

**Prerequisites** (from the CARE-PD repo, run in your shell before this notebook):
```bash
git clone https://github.com/TaatiTeam/CARE-PD.git
cd CARE-PD
pip install -r requirements.txt
huggingface-cli download vida-adl/CARE-PD --repo-type dataset --local-dir ./assets/datasets
bash scripts/preprocess_smpl2h36m.sh
```
This notebook assumes the resulting `assets/datasets/h36m/<COHORT>/` files are accessible at the path set in Section 1.

In [1]:
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter, find_peaks
from scipy.stats import kruskal, spearmanr, mannwhitneyu
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import torch
import torch.nn as nn
from umap import UMAP
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


/opt/miniconda3/envs/ma_v1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


## 1. Inspect CARE-PD's actual file structure — RUN THIS FIRST

Pick one cohort with UPDRS-gait labels (per the repo's `folds/UPDRS_Datasets/`) and load it raw, without assuming anything about its internal shape. Adjust `COHORT` and `H36M_DATA_ROOT` to your setup.

In [3]:
import numpy as np
from pathlib import Path

h36m_dir = Path("/Users/jackiebai/CARE-PD/assets/datasets/h36m/PD-GaM")
contents = sorted(h36m_dir.iterdir())
print(f"{len(contents)} items in {h36m_dir}")
for p in contents[:10]:
    print(f"  {p.name}")

# load whatever the first file actually is
sample_path = contents[0]
if sample_path.suffix == '.npy':
    sample = np.load(sample_path, allow_pickle=True)
    print(f"\n{sample_path.name}: type={type(sample)}")
    if hasattr(sample, 'shape'):
        print(f"shape={sample.shape}, dtype={sample.dtype}")
    elif isinstance(sample, dict):
        print(f"dict keys: {list(sample.keys())}")
elif sample_path.suffix == '.pkl':
    import pickle
    with open(sample_path, 'rb') as f:
        sample = pickle.load(f)
    print(f"\n{sample_path.name}: type={type(sample)}")

5 items in /Users/jackiebai/CARE-PD/assets/datasets/h36m/PD-GaM
  h36m_3d_world2cam2img_backright_floorXZZplus_30f_or_longer.npz
  h36m_3d_world2cam2img_sideright_floorXZZplus_30f_or_longer.npz
  h36m_3d_world2cam_backright_floorXZZplus_30f_or_longer.npz
  h36m_3d_world2cam_sideright_floorXZZplus_30f_or_longer.npz
  h36m_3d_world_floorXZZplus_30f_or_longer.npz


In [4]:
import numpy as np

world_path = "/Users/jackiebai/CARE-PD/assets/datasets/h36m/PD-GaM/h36m_3d_world_floorXZZplus_30f_or_longer.npz"
data = np.load(world_path, allow_pickle=True)

print(f"Keys in this npz: {data.files}")
for key in data.files:
    arr = data[key]
    print(f"\n'{key}': shape={arr.shape}, dtype={arr.dtype}")
    if arr.dtype == object:
        print(f"  first element type: {type(arr[0])}")
        if hasattr(arr[0], 'shape'):
            print(f"  first element shape: {arr[0].shape}")

Keys in this npz: ['007__007-13-000661_wid00_0', '007__007-13-000661_wid00_1', '007__007-13-000661_wid00_2', '007__007-13-000661_wid00_3', '007__007-13-000662_wid01_0', '007__007-13-000662_wid01_1', '007__007-13-000662_wid01_2', '007__007-13-000662_wid01_3', '007__007-13-001992_wid00_0', '007__007-13-001992_wid00_1', '007__007-13-001992_wid00_2', '007__007-13-001992_wid00_3', '007__007-13-002006_wid00_0', '007__007-13-002006_wid00_1', '007__007-13-002006_wid00_2', '007__007-13-002006_wid00_3', '007__007-13-003194_wid01_0', '007__007-13-003194_wid01_1', '007__007-13-003194_wid01_2', '007__007-13-003194_wid01_3', '007__007-13-003195_wid00_0', '007__007-13-003195_wid00_1', '007__007-13-003195_wid00_2', '007__007-13-003195_wid00_3', '007__007-13-004442_wid00_0', '007__007-13-004442_wid00_1', '007__007-13-004442_wid00_2', '007__007-13-004442_wid00_3', '007__007-13-004450_wid00_0', '007__007-13-004450_wid00_1', '007__007-13-004450_wid00_2', '007__007-13-004450_wid00_3', '007__007-13-005692_w

In [13]:
import re
import pickle
import numpy as np
from pathlib import Path

CARE_PD_ROOT = Path("/Users/jackiebai/CARE-PD/assets/datasets")
UPDRS_COHORTS = ["PD-GaM", "BMCLab", "3DGait", "T-SDU-PD"]


def parse_walk_id(npz_key):
    """
    Splits an npz key into (subj_id_str, walk_id, down_variant).
    Handles the '_down{N}' suffix some cohorts (e.g. BMCLab, likely due to
    its 150Hz source rate) append during conversion — walk_id returned here
    matches the raw pkl's actual walk keys; down_variant is None if absent.
    """
    subj_id_str, walk_part = npz_key.split("__", 1)
    match = re.match(r"^(.*)_down(\d+)$", walk_part)
    if match:
        return subj_id_str, match.group(1), int(match.group(2))
    return subj_id_str, walk_part, None


def resolve_subject(raw_data, subj_id_str):
    """
    Some cohorts (e.g. 3DGait) key their raw pkl by integer subject ID,
    while the npz always encodes it as a string. Try both.
    """
    if subj_id_str in raw_data:
        return subj_id_str
    try:
        as_int = int(subj_id_str)
        if as_int in raw_data:
            return as_int
    except ValueError:
        pass
    return None


sequences          = []
severity_labels    = []
medication_labels  = []
other_labels       = []
clip_names         = []
clip_cohort        = []
clip_participant   = []
clip_fps           = []
clip_walk_base_id  = []   # walk_id WITHOUT the _down suffix — groups downsampled variants of the same walk
clip_down_variant  = []   # which down-variant this is (None if not applicable)

for cohort in UPDRS_COHORTS:
    with open(CARE_PD_ROOT / f"{cohort}.pkl", "rb") as f:
        raw_data = pickle.load(f)

    npz_candidates = list((CARE_PD_ROOT / "h36m" / cohort).glob("h36m_3d_world_floorXZZplus_30f_or_longer*.npz"))
    if not npz_candidates:
        print(f"{cohort}: no world-coordinate file found — skipping")
        continue
    npz_path = npz_candidates[0]
    print(f"{cohort}: using {npz_path.name}")
    npz_data = np.load(npz_path, allow_pickle=True)

    n_matched, n_unmatched = 0, 0
    unmatched_keys = []
    for key in npz_data.files:
        subj_id_str, walk_id, down_variant = parse_walk_id(key)

        resolved_subj = resolve_subject(raw_data, subj_id_str)
        if resolved_subj is None:
            n_unmatched += 1
            unmatched_keys.append(key)
            continue

        joints = npz_data[key]
        walk_meta = raw_data.get(resolved_subj, {}).get(walk_id)
        if walk_meta is None:
            n_unmatched += 1
            unmatched_keys.append(key)
            continue
        n_matched += 1

        sequences.append(joints)
        severity_labels.append(walk_meta.get('UPDRS_GAIT'))
        medication_labels.append(walk_meta.get('medication'))
        other_labels.append(walk_meta.get('other'))
        clip_names.append(key)
        clip_cohort.append(cohort)
        clip_participant.append(f"{cohort}_{subj_id_str}")
        clip_fps.append(walk_meta.get('fps'))
        clip_walk_base_id.append(f"{cohort}_{subj_id_str}_{walk_id}")
        clip_down_variant.append(down_variant)

    print(f"  matched {n_matched}, unmatched {n_unmatched} (of {len(npz_data.files)} npz entries)")
    if unmatched_keys:
        print(f"  sample unmatched keys: {unmatched_keys[:5]}")

severity_labels = np.array([np.nan if s is None else s for s in severity_labels], dtype=float)
clip_fps = np.array(clip_fps)

print(f"\nTotal loaded: {len(sequences)} walks")
print(f"Distinct participants: {len(set(clip_participant))}")
print(f"Distinct base walks (collapsing down-variants): {len(set(clip_walk_base_id))}")
print(f"Walks with UPDRS_GAIT score: {(~np.isnan(severity_labels)).sum()} / {len(severity_labels)}")
print(f"FPS values present: {sorted(set(clip_fps))}")
if sequences:
    print(f"Sequence shape example: {sequences[0].shape}")

PD-GaM: using h36m_3d_world_floorXZZplus_30f_or_longer.npz
  matched 1700, unmatched 0 (of 1700 npz entries)
BMCLab: using h36m_3d_world_floorXZZplus_30f_or_longer.npz
  matched 3895, unmatched 0 (of 3895 npz entries)
3DGait: using h36m_3d_world_floorXZZplus_30f_or_longer.npz
  matched 90, unmatched 0 (of 90 npz entries)
T-SDU-PD: using h36m_3d_world_floorXZZplus_30f_or_longer_slopeCorrected.npz
  matched 381, unmatched 0 (of 381 npz entries)

Total loaded: 6066 walks
Distinct participants: 110
Distinct base walks (collapsing down-variants): 2950
Walks with UPDRS_GAIT score: 6066 / 6066
FPS values present: [25, 30, 150]
Sequence shape example: (95, 17, 3)


In [2]:
H36M_DATA_ROOT = Path("/Users/jackiebai/Desktop/lab/movement_analysis_RAE_project/doi-10.5683-sp3-twikmk")   # ← set this to your actual path
COHORT = "PD-GaM"   # ← swap to whichever UPDRS-labeled cohort you want to start with

cohort_dir = H36M_DATA_ROOT / COHORT
print(f"Looking in: {cohort_dir}")
print(f"Exists: {cohort_dir.exists()}")

if cohort_dir.exists():
    contents = sorted(cohort_dir.iterdir())
    print(f"\n{len(contents)} items found. First 10:")
    for p in contents[:10]:
        print(f"  {p.name}  ({'dir' if p.is_dir() else p.stat().st_size} {'​' if p.is_dir() else 'bytes'})")
else:
    # fall back to the raw per-cohort pickle at the dataset root, in case the h36m
    # conversion script writes elsewhere or wasn't run yet
    alt_path = H36M_DATA_ROOT.parent / f"{COHORT}.pkl"
    print(f"\nh36m dir not found — trying raw pickle instead: {alt_path}")
    print(f"Exists: {alt_path.exists()}")


def inspect(obj, prefix="", max_depth=4, depth=0):
    """Generic recursive structure printer — works whether CARE-PD's format
    turns out to be a dict, a list of dicts, a numpy array, or nested combinations."""
    if depth > max_depth:
        print(prefix + "...")
        return
    if isinstance(obj, dict):
        print(prefix + f"dict with {len(obj)} keys: {list(obj.keys())[:15]}")
        for k in list(obj.keys())[:5]:
            print(prefix + f"  ['{k}'] ->")
            inspect(obj[k], prefix + "    ", max_depth, depth + 1)
    elif isinstance(obj, (list, tuple)):
        print(prefix + f"{type(obj).__name__} of length {len(obj)}")
        if len(obj) > 0:
            print(prefix + "  [0] ->")
            inspect(obj[0], prefix + "    ", max_depth, depth + 1)
    elif isinstance(obj, np.ndarray):
        print(prefix + f"ndarray shape={obj.shape}, dtype={obj.dtype}")
    else:
        print(prefix + f"{type(obj).__name__}: {repr(obj)[:120]}")


# try loading whichever file/format actually exists
sample_file = None
if cohort_dir.exists():
    npy_files = list(cohort_dir.glob("*.npy"))
    pkl_files = list(cohort_dir.glob("*.pkl"))
    json_files = list(cohort_dir.glob("*.json"))
    print(f"\n.npy files: {len(npy_files)}, .pkl files: {len(pkl_files)}, .json files: {len(json_files)}")

    if pkl_files:
        sample_file = pkl_files[0]
        with open(sample_file, 'rb') as f:
            data = pickle.load(f)
    elif npy_files:
        sample_file = npy_files[0]
        data = np.load(sample_file, allow_pickle=True)
    elif json_files:
        sample_file = json_files[0]
        with open(sample_file) as f:
            data = json.load(f)
else:
    alt_path = H36M_DATA_ROOT.parent / f"{COHORT}.pkl"
    if alt_path.exists():
        sample_file = alt_path
        with open(alt_path, 'rb') as f:
            data = pickle.load(f)

if sample_file is not None:
    print(f"\n=== Structure of {sample_file.name} ===")
    inspect(data)
else:
    print("\nNo file found automatically — set the path manually and load it yourself, "
          "then call inspect(your_loaded_object) to see its structure.")


Looking in: /Users/jackiebai/Desktop/lab/movement_analysis_RAE_project/doi-10.5683-sp3-twikmk/PD-GaM
Exists: False

h36m dir not found — trying raw pickle instead: /Users/jackiebai/Desktop/lab/movement_analysis_RAE_project/PD-GaM.pkl
Exists: False

No file found automatically — set the path manually and load it yourself, then call inspect(your_loaded_object) to see its structure.


## 2. Load all UPDRS-labeled cohorts into `sequences` / `severity_labels` / `clip_names`

**This cell makes assumptions about the data shape based on Section 1's output — edit the loop body to match what you actually saw.** The scaffold below assumes each cohort file yields a dict or list of per-walk entries, each with a `(T, 17, 3)` H36M-joint array and an UPDRS-gait score where available (per the repo's `UPDRS_Datasets` fold). Adjust the key names (`'keypoints'`, `'updrs_gait'`, `'participant_id'`, etc.) once Section 1 shows you the real ones.

In [ ]:
UPDRS_COHORTS = ["PD-GaM"]   # ← add every cohort listed under folds/UPDRS_Datasets/ once confirmed

sequences        = []   # list of (T, 17, 3) arrays
severity_labels  = []   # UPDRS-gait score, or np.nan if unavailable for that walk
clip_names       = []
clip_cohort      = []
clip_participant = []

for cohort in UPDRS_COHORTS:
    cohort_dir = H36M_DATA_ROOT / cohort
    pkl_path = cohort_dir / f"{cohort}.pkl" if cohort_dir.exists() else H36M_DATA_ROOT.parent / f"{cohort}.pkl"

    with open(pkl_path, 'rb') as f:
        cohort_data = pickle.load(f)

    # ── ADJUST THIS LOOP to match Section 1's actual structure ──
    # placeholder assumes cohort_data is a list of dicts; common alternative is a
    # dict keyed by walk/participant id — check Section 1's printout and rewrite
    # this iteration accordingly (e.g. `for walk_id, entry in cohort_data.items():`)
    entries = cohort_data if isinstance(cohort_data, list) else cohort_data.values()

    for i, entry in enumerate(entries):
        keypoints = entry.get('keypoints_3d', entry.get('keypoints', entry.get('joints_3d')))
        if keypoints is None:
            continue
        keypoints = np.asarray(keypoints, dtype=np.float32)
        if keypoints.ndim == 2:  # (T, joints*3) flattened -> reshape
            n_joints = keypoints.shape[1] // 3
            keypoints = keypoints.reshape(-1, n_joints, 3)

        updrs = entry.get('updrs_gait', entry.get('UPDRS_gait', entry.get('updrs_score', np.nan)))
        participant = entry.get('participant_id', entry.get('subject_id', f"{cohort}_{i}"))

        sequences.append(keypoints)
        severity_labels.append(float(updrs) if updrs is not None else np.nan)
        clip_names.append(f"{cohort}_{i}")
        clip_cohort.append(cohort)
        clip_participant.append(participant)

severity_labels = np.array(severity_labels, dtype=float)

print(f"Loaded {len(sequences)} walks across {len(UPDRS_COHORTS)} cohort(s)")
print(f"Walks with a resolved UPDRS-gait score: {(~np.isnan(severity_labels)).sum()} / {len(severity_labels)}")
print(f"Distinct participants: {len(set(clip_participant))}")
if sequences:
    print(f"Sequence shape example: {sequences[0].shape}")
    print(f"Sequence lengths — min: {min(len(s) for s in sequences)}, "
          f"max: {max(len(s) for s in sequences)}, mean: {np.mean([len(s) for s in sequences]):.1f}")


## 3. Confirm H36M joint ordering, then define joint indices

Standard Human3.6M 17-joint order (used by MotionBERT, PoseFormerV2, and most of the H36M-format literature CARE-PD's own repo cites): `0=Hip(root/pelvis), 1=RHip, 2=RKnee, 3=RFoot, 4=LHip, 5=LKnee, 6=LFoot, 7=Spine, 8=Thorax, 9=Neck/Nose, 10=Head, 11=LShoulder, 12=LElbow, 13=LWrist, 14=RShoulder, 15=RElbow, 16=RWrist`. **Verify this against CARE-PD's own documentation (`docs/dataset.md`) before trusting it** — different H36M-format pipelines occasionally reorder or omit joints, and this is exactly the kind of assumption that silently produces garbage if wrong, the same way a swapped `joint_dim`/`num_joints` did earlier in this project.

In [ ]:
H36M_JOINT_NAMES = [
    'Hip', 'RHip', 'RKnee', 'RFoot', 'LHip', 'LKnee', 'LFoot',
    'Spine', 'Thorax', 'Neck_Nose', 'Head',
    'LShoulder', 'LElbow', 'LWrist', 'RShoulder', 'RElbow', 'RWrist'
]
N_JOINTS = len(H36M_JOINT_NAMES)  # 17

HIP     = H36M_JOINT_NAMES.index('Hip')
LHIP    = H36M_JOINT_NAMES.index('LHip')
RHIP    = H36M_JOINT_NAMES.index('RHip')
LSHO    = H36M_JOINT_NAMES.index('LShoulder')
RSHO    = H36M_JOINT_NAMES.index('RShoulder')
HEAD    = H36M_JOINT_NAMES.index('Head')

# sanity check against loaded data
if sequences:
    assert sequences[0].shape[1] == N_JOINTS, \
        f"Loaded data has {sequences[0].shape[1]} joints, expected {N_JOINTS} — check H36M joint count"
    print(f"✓ Joint count matches: {N_JOINTS}")


## 4. 3D preprocessing — hip-centered, torso-normalized, YAW-ONLY heading alignment

Adapted from `preprocess_human_normalized`. Key change from the 2D version: heading is computed and corrected only in the horizontal plane (x-z, assuming H36M's y-axis is vertical/up — **verify this convention against CARE-PD's docs**, since it depends on how their SMPL-to-H36M conversion oriented the coordinate frame). The up-axis itself is never rotated, unlike the 2D version which rotated the whole plane.

In [ ]:
UP_AXIS = 1  # ← VERIFY: standard H36M convention is often y-up (axis index 1). Check a still frame
             #   (Section 9 below) before trusting this — if the skeleton looks sideways or upside
             #   down, this is very likely the wrong axis.
HORIZ_AXES = [a for a in range(3) if a != UP_AXIS]  # the two horizontal axes for heading/rotation


def preprocess_3d_normalized(raw_sequence):
    """
    raw_sequence: (T, 17, 3) H36M-ordered joint positions.
    Returns (T, 17, 10): position(3) + velocity(3) + yaw_angular_velocity(1) + speed(1) + sin/cos_yaw(2)
    """
    left_hip  = raw_sequence[:, LHIP, :]
    right_hip = raw_sequence[:, RHIP, :]
    center    = (left_hip + right_hip) / 2.0

    center_diff = np.diff(center, axis=0)
    center_diff = np.concatenate([center_diff[:1], center_diff], axis=0)
    speed       = np.linalg.norm(center_diff, axis=-1)

    centered = raw_sequence - center[:, np.newaxis, :]

    left_shoulder  = centered[:, LSHO, :]
    right_shoulder = centered[:, RSHO, :]
    shoulder_mid   = (left_shoulder + right_shoulder) / 2.0
    torso_length   = np.linalg.norm(shoulder_mid, axis=-1)
    mean_torso     = torso_length.mean()

    if mean_torso > 1e-6:
        centered = centered / mean_torso
        speed    = speed / mean_torso

    speed = np.tile(speed[:, None, None], (1, N_JOINTS, 1))

    # yaw heading from head position, projected onto the horizontal plane only
    head_pt = centered[:, HEAD, :]
    h0, h1 = HORIZ_AXES
    angle = np.arctan2(head_pt[:, h1], head_pt[:, h0])

    cos_a = np.cos(-angle)
    sin_a = np.sin(-angle)

    aligned = centered.copy()
    horiz0 = centered[:, :, h0]
    horiz1 = centered[:, :, h1]
    aligned[:, :, h0] = horiz0 * cos_a[:, None] - horiz1 * sin_a[:, None]
    aligned[:, :, h1] = horiz0 * sin_a[:, None] + horiz1 * cos_a[:, None]
    # aligned[:, :, UP_AXIS] stays exactly as in `centered` — never rotated

    vel = np.diff(aligned, axis=0)
    vel = np.concatenate([vel[:1], vel], axis=0)

    ang_vel = np.diff(angle, axis=0)
    ang_vel = np.concatenate([ang_vel[:1], ang_vel])
    ang_vel = np.tile(ang_vel[:, None, None], (1, N_JOINTS, 1))

    sin_h = np.tile(np.sin(angle)[:, None, None], (1, N_JOINTS, 1))
    cos_h = np.tile(np.cos(angle)[:, None, None], (1, N_JOINTS, 1))

    return np.concatenate([aligned, vel, ang_vel, speed, sin_h, cos_h], axis=-1)  # (T, 17, 10)


## 5. Build `raw_processed`, `clip_frame_ranges`, `severity_per_frame`

In [ ]:
all_processed        = []
clip_frame_ranges    = []
severity_per_frame_l = []
cursor = 0

for seq, sev in zip(sequences, severity_labels):
    if len(seq) < 5:
        continue
    processed = preprocess_3d_normalized(seq)
    processed[0, :, 3:] = 0.0   # zero derivative-based channels on frame 0 (position channels are 0:3)
    all_processed.append(processed)
    clip_frame_ranges.append((cursor, cursor + len(processed)))
    severity_per_frame_l.extend([sev] * len(processed))
    cursor += len(processed)

raw_processed       = np.concatenate(all_processed, axis=0)
severity_per_frame  = np.array(severity_per_frame_l)

print(f"raw_processed shape: {raw_processed.shape}")
print(f"Frames with a resolved UPDRS-gait score: {(~np.isnan(severity_per_frame)).sum()} / {len(severity_per_frame)}")


## 6. Scale features

In [ ]:
n_frames, n_joints, n_coords = raw_processed.shape
da_scaled = np.zeros_like(raw_processed)
scalers   = []

for j in range(n_joints):
    joint_scalers = []
    for c in range(n_coords):
        channel = raw_processed[:, j, c].reshape(-1, 1)
        scaler  = StandardScaler()
        normed  = scaler.fit_transform(channel)
        normed  = np.clip(normed, -5.0, 5.0)
        std     = normed.std()
        if std > 1e-8:
            normed = normed / std
        normed  = np.clip(normed, -5.0, 5.0)
        da_scaled[:, j, c] = normed.squeeze()
        joint_scalers.append(scaler)
    scalers.append(joint_scalers)

raw_processed = da_scaled
print(f"Scaled shape: {raw_processed.shape}")


## 7. Verify the 3D skeleton looks right BEFORE training anything

Same discipline as the LA/REMAP skeleton checks earlier in this project — confirm the up-axis assumption and joint ordering visually before spending compute on a model trained on garbage.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

H36M_EDGES = [
    (HIP, RHIP), (RHIP, 2), (2, 3),
    (HIP, LHIP), (LHIP, 5), (5, 6),
    (HIP, 7), (7, 8), (8, 9), (9, 10),
    (8, LSHO), (LSHO, 12), (12, 13),
    (8, RSHO), (RSHO, 15), (15, 16),
]

FRAME = 0
fig = plt.figure(figsize=(7, 8))
ax = fig.add_subplot(111, projection='3d')

frame_xyz = raw_processed[FRAME, :, 0:3]  # position channels, pre-scaling would be clearer — see note below

for j1, j2 in H36M_EDGES:
    ax.plot([frame_xyz[j1,0], frame_xyz[j2,0]],
            [frame_xyz[j1,1], frame_xyz[j2,1]],
            [frame_xyz[j1,2], frame_xyz[j2,2]], 'gray')
for j, name in enumerate(H36M_JOINT_NAMES):
    x, y, z = frame_xyz[j]
    ax.scatter(x, y, z, s=40)
    ax.text(x, y, z, name, fontsize=7)

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title(f"Frame {FRAME} — check this looks like an upright human before proceeding")
plt.tight_layout()
plt.show()

# NOTE: this plots the SCALED raw_processed (post-Section 6). If the skeleton looks
# distorted, re-run this against an unscaled copy saved before Section 6 to isolate
# whether the issue is UP_AXIS/joint-order (fix in Section 3/4) or the scaling step.


## 8. Hyperparameters, model, helper functions

In [ ]:
MAX_ITER = 30
EPOCHS = 2000
VW_EPOCHS = 1000
BATCH_SIZE = 64
LR = 1e-3
WINDOW_SIZE = 30
LATENT_DIM = 64
FPS = 30   # ← confirm CARE-PD's actual capture/resample rate in their docs

MIN_SEGMENT_FRAMES = 20
MAX_SEGMENT_FRAMES = 300
MIN_WINDOWS = 150

PERCENTILE_RANGE = (20, 80)
QUANTILE_RANGE   = (0.10, 0.30)


In [ ]:
class HierarchicalRAE(nn.Module):
    def __init__(self, joint_dim, joint_embed=32, pose_embed=128,
                 hidden_dim=256, latent_dim=LATENT_DIM, num_joints=N_JOINTS):
        super().__init__()
        self.num_joints = num_joints
        self.joint_encoder = nn.Sequential(
            nn.Linear(joint_dim, joint_embed), nn.Tanh(), nn.Linear(joint_embed, joint_embed))
        self.pose_encoder = nn.Sequential(
            nn.Linear(num_joints * joint_embed, pose_embed), nn.Tanh())
        self.encoder_rnn = nn.LSTM(pose_embed, hidden_dim, batch_first=True)
        self.fc_latent = nn.Linear(hidden_dim, latent_dim)
        self.fc_decode_h = nn.Linear(latent_dim, hidden_dim)
        self.fc_decode_c = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.decoder_proj = nn.Linear(hidden_dim, pose_embed)
        self.pose_decoder = nn.Sequential(
            nn.Linear(pose_embed, num_joints * joint_embed), nn.Tanh())
        self.joint_decoder = nn.Linear(joint_embed, joint_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc_decode_h.weight, gain=2.0)
        nn.init.xavier_uniform_(self.fc_decode_c.weight, gain=2.0)
        nn.init.constant_(self.fc_decode_h.bias, 0.0)
        nn.init.constant_(self.fc_decode_c.bias, 0.0)
        nn.init.xavier_uniform_(self.fc_latent.weight, gain=2.0)
        nn.init.constant_(self.fc_latent.bias, 0.0)
        for name, param in self.decoder_rnn.named_parameters():
            if 'weight_ih' in name: nn.init.xavier_uniform_(param, gain=2.0)
            elif 'weight_hh' in name: nn.init.orthogonal_(param, gain=2.0)
            elif 'bias' in name: nn.init.zeros_(param)
        for name, param in self.encoder_rnn.named_parameters():
            if 'weight_ih' in name: nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name: nn.init.orthogonal_(param)
            elif 'bias' in name: nn.init.zeros_(param)
        for module in [self.joint_encoder, self.pose_encoder, self.pose_decoder,
                       self.joint_decoder, self.decoder_proj]:
            if isinstance(module, nn.Sequential):
                for layer in module:
                    if isinstance(layer, nn.Linear):
                        nn.init.xavier_uniform_(layer.weight); nn.init.zeros_(layer.bias)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight); nn.init.zeros_(module.bias)

    def encode(self, x, lengths=None):
        B, T, J, C = x.shape
        x_flat = x.view(B * T * J, C)
        x_flat = self.joint_encoder(x_flat)
        x_enc  = x_flat.view(B, T, J, -1).view(B, T, -1)
        x_enc  = self.pose_encoder(x_enc)
        if lengths is not None:
            packed = pack_padded_sequence(x_enc, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, (h, _) = self.encoder_rnn(packed)
        else:
            _, (h, _) = self.encoder_rnn(x_enc)
        return self.fc_latent(h[-1])

    def decode(self, z, T):
        h = self.fc_decode_h(z).unsqueeze(0)
        c = self.fc_decode_c(z).unsqueeze(0)
        inp = z.unsqueeze(1).repeat(1, T, 1)
        dec, _ = self.decoder_rnn(inp, (h, c))
        return self.decoder_proj(dec)

    def forward(self, x, lengths=None):
        B, T, J, C = x.shape
        z = self.encode(x, lengths)
        dec = self.decode(z, T)
        dec = self.pose_decoder(dec).view(B, T, J, -1).view(B * T * J, -1)
        dec = self.joint_decoder(dec).view(B, T, J, C)
        return dec, z


def collate_variable_length(batch):
    lengths = [x.shape[0] for x in batch]
    B, T_max = len(batch), max(lengths)
    J, C = batch[0].shape[1], batch[0].shape[2]
    padded = torch.zeros(B, T_max, J, C)
    for i, x in enumerate(batch):
        padded[i, :lengths[i]] = x
    return padded, torch.tensor(lengths, dtype=torch.long)


def extract_nonoverlapping_windows(raw_sequence, window_size):
    n_frames = len(raw_sequence)
    windows = np.array([raw_sequence[s:s+window_size] for s in range(0, n_frames - window_size, window_size)])
    print(f"Extracted {len(windows)} non-overlapping windows")
    return windows


def train_on_fixed_windows(raw_sequence, window_size=WINDOW_SIZE, epochs=EPOCHS,
                            batch_size=BATCH_SIZE, lr=LR, device=device, patience=30):
    windows = extract_nonoverlapping_windows(raw_sequence, window_size)
    X_tensor = torch.tensor(windows, dtype=torch.float32)
    n_joints, joint_dim = raw_sequence.shape[1], raw_sequence.shape[2]
    from torch.utils.data import TensorDataset
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=True)

    model = HierarchicalRAE(latent_dim=LATENT_DIM, joint_dim=joint_dim, num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    loss_fn = nn.MSELoss()
    best_loss, patience_ctr, best_weights, lossgraph = float('inf'), 0, None, []

    for epoch in range(epochs):
        total_loss = 0
        for (batch,) in loader:
            batch = batch.to(device)
            recon, _ = model(batch)
            loss = loss_fn(recon, batch)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        avg = total_loss / len(loader)
        lossgraph.append(avg); scheduler.step(avg)
        if avg < best_loss - 1e-4:
            best_loss, patience_ctr = avg, 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights); break
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}")
    return model, lossgraph


def train_on_variable_windows(raw_sequence, windows, epochs=VW_EPOCHS, batch_size=BATCH_SIZE,
                               lr=LR, device=device, patience=30):
    n_joints, joint_dim = raw_sequence.shape[1], raw_sequence.shape[2]
    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=True, collate_fn=collate_variable_length)
    model = HierarchicalRAE(latent_dim=LATENT_DIM, joint_dim=joint_dim, num_joints=n_joints).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    loss_fn = nn.MSELoss()
    best_loss, patience_ctr, best_weights, lossgraph = float('inf'), 0, None, []

    for epoch in range(epochs):
        total_loss = 0
        for padded, lengths in loader:
            padded = padded.to(device)
            recon, _ = model(padded, lengths=lengths)
            loss = torch.tensor(0.0, device=device)
            for i, l in enumerate(lengths):
                loss = loss + loss_fn(recon[i, :l], padded[i, :l])
            loss = loss / len(lengths)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        avg = total_loss / len(loader)
        lossgraph.append(avg); scheduler.step(avg)
        if avg < best_loss - 1e-4:
            best_loss, patience_ctr = avg, 0
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1} — best loss: {best_loss:.6f}")
            model.load_state_dict(best_weights); break
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {avg:.6f}")
    return model, lossgraph


def compute_frame_loss(model, raw_sequence, window_size, stride=5, device=device):
    model.eval()
    losses, positions = [], []
    raw_tensor = torch.tensor(raw_sequence, dtype=torch.float32, device=device)
    loss_fn = nn.MSELoss()
    with torch.no_grad():
        for start in range(0, len(raw_sequence) - window_size, stride):
            window = raw_tensor[start:start + window_size].unsqueeze(0)
            recon, _ = model(window)
            losses.append(loss_fn(recon, window).item())
            positions.append(start + window_size // 2)
    positions, losses = np.array(positions), np.array(losses)
    print(f"Computed loss at {len(losses)} positions")
    return positions, losses


def find_transitions(positions, losses, percentile=70, smoothing=5, min_distance=3, fps=FPS):
    if len(losses) < smoothing:
        smoothing = max(3, len(losses) // 2)
        if smoothing % 2 == 0: smoothing += 1
    smoothed = savgol_filter(losses, window_length=smoothing, polyorder=2)
    threshold = np.percentile(smoothed, percentile)
    peaks, _ = find_peaks(smoothed, height=threshold, distance=min_distance)
    transition_frames = positions[peaks]
    if len(transition_frames) > 1:
        mean_bout = np.mean(np.diff(transition_frames)) / fps
        print(f"Found {len(transition_frames)} transitions, mean bout duration: {mean_bout:.2f}s")
    return transition_frames, smoothed


def create_windows_from_transitions(raw_sequence, transition_frames,
                                     min_segment_frames=MIN_SEGMENT_FRAMES, max_segment_frames=MAX_SEGMENT_FRAMES):
    n_frames = len(raw_sequence)
    boundaries = np.unique(np.concatenate([[0], transition_frames, [n_frames]])).astype(int)
    all_windows, window_labels, window_starts = [], [], []
    for seg_idx in range(len(boundaries) - 1):
        seg_start, seg_end = boundaries[seg_idx], boundaries[seg_idx + 1]
        seg_len = seg_end - seg_start
        if seg_len < min_segment_frames:
            continue
        segment = raw_sequence[seg_start:seg_end]
        for start in range(0, seg_len, max_segment_frames):
            chunk = segment[start:start + max_segment_frames]
            if len(chunk) < min_segment_frames:
                continue
            all_windows.append(chunk)
            window_labels.append(seg_idx)
            window_starts.append(seg_start + start)
    lengths = [len(w) for w in all_windows]
    print(f"Created {len(all_windows)} variable-length windows from {len(boundaries) - 1} segments")
    if lengths:
        print(f"Window lengths — min: {min(lengths)}, max: {max(lengths)}, mean: {np.mean(lengths):.1f}")
    return all_windows, np.array(window_labels), np.array(window_starts)


def create_windows_from_transitions_clip_safe(raw_sequence, transition_frames, clip_frame_ranges,
                                                min_segment_frames=MIN_SEGMENT_FRAMES,
                                                max_segment_frames=MAX_SEGMENT_FRAMES):
    """
    Same clip-boundary-safety fix used for REMAP: forces a hard cut at every
    walk's start frame, so no window can span more than one walk/participant
    recording, regardless of the loss signal.
    """
    clip_starts = np.array([start for start, end in clip_frame_ranges])
    all_boundaries = np.unique(np.concatenate([transition_frames, clip_starts]))
    return create_windows_from_transitions(raw_sequence, all_boundaries,
                                            min_segment_frames, max_segment_frames)


def frame_to_clip(frame_idx):
    for clip_i, (start, end) in enumerate(clip_frame_ranges):
        if start <= frame_idx < end:
            return clip_i
    return -1


def encode_and_cluster(model, windows, batch_size=BATCH_SIZE, device=device, quantile=0.1,
                        umap_neighbors=30, umap_min_dist=0.1):
    model.eval()
    all_latents = []
    tensor_list = [torch.tensor(w, dtype=torch.float32) for w in windows]
    loader = DataLoader(tensor_list, batch_size=batch_size, shuffle=False, collate_fn=collate_variable_length)
    with torch.no_grad():
        for padded, lengths in loader:
            padded = padded.to(device)
            _, z = model(padded, lengths=lengths)
            all_latents.append(z.cpu().numpy())
    all_latents = np.concatenate(all_latents, axis=0)
    print(f"Latents shape: {all_latents.shape}")

    reducer = UMAP(n_components=2, n_neighbors=umap_neighbors, min_dist=umap_min_dist,
                  metric='cosine', random_state=42)
    latents_2d = reducer.fit_transform(all_latents)
    print(f"UMAP done. Shape: {latents_2d.shape}")

    normed = normalize(latents_2d, norm='l2')
    bandwidth = estimate_bandwidth(normed, quantile=quantile)
    ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    ms.fit(normed)
    cluster_labels = ms.labels_
    n_clusters = len(np.unique(cluster_labels))
    print(f"Found {n_clusters} clusters, sizes: {np.bincount(cluster_labels)}")

    if n_clusters > 1:
        dist_matrix = np.clip(1 - np.dot(normed, normed.T), 0, 2)
        sil = silhouette_score(dist_matrix, cluster_labels, metric='precomputed')
        print(f"Silhouette score: {sil:.4f}")

    return all_latents, latents_2d, cluster_labels, ms


## 9. Stage 1 training + reconstruction loss

In [ ]:
model_stage1, lossgraph_stage1 = train_on_fixed_windows(
    raw_processed, window_size=WINDOW_SIZE, epochs=EPOCHS, device=device
)
torch.save(model_stage1.state_dict(), 'carepd_model_stage1.pth')

plt.figure(figsize=(10, 4))
plt.plot(lossgraph_stage1)
plt.title('Stage 1 Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.show()

positions, losses = compute_frame_loss(model_stage1, raw_processed, window_size=WINDOW_SIZE, stride=2, device=device)


## 10. Verify clip-safe windowing before searching

In [ ]:
test_transitions, _ = find_transitions(positions, losses, percentile=50, fps=FPS)
test_windows, _, test_starts = create_windows_from_transitions_clip_safe(
    raw_processed, test_transitions, clip_frame_ranges
)

multi_clip_count = 0
for w in range(len(test_windows)):
    frame_clip_ids = [frame_to_clip(test_starts[w] + i) for i in range(len(test_windows[w]))]
    if len(set(c for c in frame_clip_ids if c != -1)) > 1:
        multi_clip_count += 1
print(f"Windows: {len(test_windows)}, multi-clip windows: {multi_clip_count} (should be 0)")


## 11. Bayesian search — score by variance explained (epsilon²) on UPDRS-gait severity

UPDRS-gait is continuous, like LA's UPDRS 3.8 leg-agility score — so epsilon²/Kruskal-Wallis (the metric that worked out best for continuous severity earlier in this project) is the natural fit here, not ARI/NMI (built for categorical labels).

In [ ]:
search_space = [
    Real(*PERCENTILE_RANGE, name='percentile'),
    Real(*QUANTILE_RANGE, name='quantile'),
]
search_log = []
best_results = None
best_score = -1
iteration = 0
MIN_PARTICIPANTS_PER_CLUSTER = 3

clip_participant_arr = np.array(clip_participant)

@use_named_args(search_space)
def objective(percentile, quantile):
    global iteration, best_score, best_results
    iteration += 1
    print(f"\n{'='*60}\nITER {iteration}/{MAX_ITER} | percentile={percentile:.2f}, quantile={quantile:.4f}\n{'='*60}")

    transition_frames, smoothed = find_transitions(positions, losses, percentile=percentile, fps=FPS)
    windows, _wl, window_starts = create_windows_from_transitions_clip_safe(
        raw_processed, transition_frames, clip_frame_ranges
    )
    if len(windows) < MIN_WINDOWS:
        print(f"  only {len(windows)} windows (<{MIN_WINDOWS}) — skipping")
        return 0.0

    window_clip_id = np.array([frame_to_clip(window_starts[w]) for w in range(len(windows))])
    window_severity = np.array([severity_labels[c] if c != -1 else np.nan for c in window_clip_id])
    window_participant = np.array([clip_participant_arr[c] if c != -1 else None for c in window_clip_id], dtype=object)

    val_mask = ~np.isnan(window_severity)
    print(f"  windows with resolved severity: {val_mask.sum()} / {len(windows)}")

    model_file = f"carepd_model_{iteration}.pth"
    if Path(model_file).is_file():
        n_joints, joint_dim = raw_processed.shape[1], raw_processed.shape[2]
        checkpoint = torch.load(model_file, weights_only=True)
        saved_latent_dim = checkpoint['fc_latent.weight'].shape[0]
        model = HierarchicalRAE(latent_dim=saved_latent_dim, joint_dim=joint_dim, num_joints=n_joints).to(device)
        model.load_state_dict(checkpoint)
    else:
        model, _ = train_on_variable_windows(raw_processed, windows, epochs=VW_EPOCHS, lr=LR, device=device)
        torch.save(model.state_dict(), model_file)

    latents, latents_2d, cluster_labels, ms_model = encode_and_cluster(
        model, windows, quantile=quantile, umap_neighbors=15, umap_min_dist=0.1, device=device
    )
    n_clusters = len(np.unique(cluster_labels))

    if n_clusters < 2 or n_clusters >= len(latents_2d):
        print(f"  → {n_clusters} clusters — invalid, skipping")
        eps_sq = 0.0
    else:
        df = pd.DataFrame({'cluster': cluster_labels[val_mask], 'severity': window_severity[val_mask],
                            'participant': window_participant[val_mask]})
        diversity = df.groupby('cluster')['participant'].nunique()
        valid_clusters = diversity[diversity >= MIN_PARTICIPANTS_PER_CLUSTER].index
        df_valid = df[df['cluster'].isin(valid_clusters)]

        if len(df_valid) < 20 or df_valid['cluster'].nunique() < 2:
            print("  not enough valid (multi-participant) clusters — skipping")
            eps_sq = 0.0
        else:
            groups = [g['severity'].values for _, g in df_valid.groupby('cluster')]
            h_stat, p_val = kruskal(*[g for g in groups if len(g) > 0])
            n_tot, k = len(df_valid), len(groups)
            eps_sq = max(0.0, (h_stat - k + 1) / (n_tot - k)) if n_tot > k else 0.0
            print(f"  → {n_clusters} clusters ({len(valid_clusters)} valid), epsilon²={eps_sq:.4f}, p={p_val:.4g}")

    search_log.append({'iteration': iteration, 'percentile': round(percentile, 2), 'quantile': round(quantile, 4),
                        'n_clusters': n_clusters, 'epsilon_sq': round(float(eps_sq), 4)})

    if eps_sq > best_score:
        best_score = eps_sq
        best_results = {
            'model': model, 'model_stage1': model_stage1, 'latents': latents, 'latents_2d': latents_2d,
            'cluster_labels': cluster_labels, 'windows': windows, 'window_starts': window_starts,
            '_percentile': percentile, '_quantile': quantile,
        }
        print(f"  ** NEW BEST (epsilon²={eps_sq:.4f}) **")

    del model, latents, latents_2d, cluster_labels, ms_model
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return -eps_sq

bayes_result = gp_minimize(func=objective, dimensions=search_space,
                            n_calls=MAX_ITER, n_initial_points=min(MAX_ITER, 5), random_state=42)
print(f"\nBest epsilon² = {best_score:.4f}")
results = best_results


## 12. Cluster vs. UPDRS-gait severity — per-cluster summary + subject diversity

In [ ]:
cluster_labels = results['cluster_labels']
windows = results['windows']
starts = results['window_starts']

window_clip_id = np.array([frame_to_clip(starts[w]) for w in range(len(windows))])
window_severity = np.array([severity_labels[c] if c != -1 else np.nan for c in window_clip_id])
window_participant = np.array([clip_participant_arr[c] if c != -1 else None for c in window_clip_id], dtype=object)

val_mask = ~np.isnan(window_severity)
summary_df = pd.DataFrame({'cluster': cluster_labels[val_mask], 'severity': window_severity[val_mask],
                            'participant': window_participant[val_mask]})

print("Per-cluster UPDRS-gait severity:")
print(summary_df.groupby('cluster')['severity'].agg(['count', 'mean', 'median', 'std']).round(3))

print("\nDistinct participants per cluster:")
print(summary_df.groupby('cluster')['participant'].nunique())

fig, ax = plt.subplots(figsize=(1.5 * summary_df['cluster'].nunique() + 3, 5))
sns.boxplot(data=summary_df, x='cluster', y='severity', hue='cluster', palette='viridis', legend=False, ax=ax)
sns.stripplot(data=summary_df, x='cluster', y='severity', color='black', alpha=0.3, size=3, ax=ax)
ax.set_title('UPDRS-gait severity by cluster')
plt.tight_layout()
plt.show()
